# PHY YOLO Pose Train

현재 수집 포맷의 port bbox와 4개 corner keypoint를 Ultralytics YOLO pose로 학습합니다.

- images: `images/<split>/<camera>/trial_<index>/*`
- annotations: `annotations/<split>/<camera>/trial_<index>/*.txt`
- metadata: `samples.jsonl`
- dataset config: `yolo_pose.yaml` (`kpt_shape: [4, 3]`)
- keypoint order: top-left, top-right, bottom-right, bottom-left

Class ID를 notebook에 고정하지 않고 `yolo_pose.yaml#names` 전체를 사용합니다. 따라서 SFP뿐 아니라 `sc_port` annotation이 추가되어도 같은 notebook으로 학습할 수 있습니다.

Local에서는 현재 `ws_aic` 경로를 사용하고, Kaggle에서는 `/kaggle/working` 아래에서 dataset과 model을 관리합니다. Kaggle Notebook Settings에서 Accelerator를 `GPU T4 x2`로 선택하면 두 GPU를 모두 사용합니다. TPU는 지원 대상이 아닙니다.

In [2]:
import importlib.util
import json
import os
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").is_dir()
required_packages = {
    "hf_xet": "hf_xet",
    "huggingface_hub": "huggingface_hub",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "ultralytics": "ultralytics",
    "yaml": "PyYAML",
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else :
    print("All required packages are already installed.")

import torch
import yaml


def find_src_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pixi.toml").is_file() and (path / "phy").is_dir():
            return path
        nested = path / "ws_aic" / "src"
        if (nested / "pixi.toml").is_file() and (nested / "phy").is_dir():
            return nested
    raise RuntimeError("ws_aic/src root를 찾지 못했습니다.")


if IS_KAGGLE:
    SRC_ROOT = Path("/kaggle/working")
    WS_ROOT = SRC_ROOT
else:
    SRC_ROOT = find_src_root(Path.cwd())
    WS_ROOT = SRC_ROOT.parent
DATASET_DIR = WS_ROOT / "data" / "img2pos" / "phy_approach"
DATA_YAML = DATASET_DIR / "yolo_pose.yaml"
SAMPLES_JSONL = DATASET_DIR / "samples.jsonl"

MODEL_NAME = "yolo11s-pose.pt"
MODEL_ROOT = WS_ROOT / "model" / "phy_yolo_pose"
RUN_NAME = "board_view"

EPOCHS = 10
IMGSZ = 640
BATCH = 16
WORKERS = 8
PATIENCE = 20
IMAGE_EXTS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

CUDA_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
TORCH_CUDA_ARCHES = set(torch.cuda.get_arch_list())
CUDA_ARCH_BY_DEVICE = {}
for index in range(CUDA_COUNT):
    major, minor = torch.cuda.get_device_capability(index)
    CUDA_ARCH_BY_DEVICE[index] = f"sm_{major}{minor}"
CUDA_DEVICES = [index for index, arch in CUDA_ARCH_BY_DEVICE.items() if arch in TORCH_CUDA_ARCHES]
TRAIN_DEVICE = CUDA_DEVICES[:2] if len(CUDA_DEVICES) >= 2 else (CUDA_DEVICES[0] if CUDA_DEVICES else "cpu")
if IS_KAGGLE and CUDA_COUNT >= 2:
    assert TRAIN_DEVICE == [0, 1], f"Kaggle T4 x2 is not supported by this PyTorch build: {CUDA_ARCH_BY_DEVICE}"
INFERENCE_DEVICE = TRAIN_DEVICE[0] if isinstance(TRAIN_DEVICE, list) else TRAIN_DEVICE
if isinstance(TRAIN_DEVICE, list) and BATCH % len(TRAIN_DEVICE):
    raise ValueError(f"BATCH={BATCH} must be divisible by GPU count={len(TRAIN_DEVICE)}")


CACHE_ROOT = WS_ROOT / ".cache"
os.environ.setdefault("HF_HOME", str(CACHE_ROOT / "huggingface"))
os.environ.setdefault("HF_HUB_CACHE", str(CACHE_ROOT / "huggingface" / "hub"))
os.environ.setdefault("YOLO_CONFIG_DIR", str(CACHE_ROOT / "ultralytics"))

RUNTIME = "kaggle" if IS_KAGGLE else "local"
print(f"runtime: {RUNTIME}")
print(f"SRC_ROOT: {SRC_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DATA_YAML: {DATA_YAML}")
print(f"TRAIN_DEVICE: {TRAIN_DEVICE}")
for index in range(CUDA_COUNT):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}, architecture={CUDA_ARCH_BY_DEVICE[index]}, supported={index in CUDA_DEVICES}")

All required packages are already installed.
runtime: local
SRC_ROOT: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/src
DATASET_DIR: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/data/img2pos/phy_approach
DATA_YAML: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/data/img2pos/phy_approach/yolo_pose.yaml
TRAIN_DEVICE: cpu
GPU 0: NVIDIA GeForce GTX 1050, architecture=sm_61, supported=False


## Hugging Face dataset download

`team-physic/aic-align`의 `260812` revision을 `DATASET_DIR`에 병렬 다운로드합니다. 재실행하면 이미 받은 파일은 재사용합니다. Private dataset이면 Local에서는 `HF_TOKEN` 또는 `hf auth login`, Kaggle에서는 Secrets의 `HF_TOKEN`을 사용합니다.

In [ ]:
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")
if IS_KAGGLE and not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

downloaded_dataset = snapshot_download(
    repo_id="team-physic/aic-align",
    repo_type="dataset",
    revision="260812",
    local_dir=DATASET_DIR,
    token=HF_TOKEN or None,
    max_workers=16,
)
print(f"Downloaded dataset: {downloaded_dataset}")

/home/swlinux/Desktop/workspace/CJ-Logistics-Challenge/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset config

Local에서는 `ws_aic/data/img2pos/phy_approach`, Kaggle에서는 `/kaggle/working/data/img2pos/phy_approach`의 `yolo_pose.yaml`을 읽습니다. 원본 `annotations`를 Ultralytics의 `labels` 규칙에 연결하는 학습용 상대 symlink view만 model directory에 만듭니다.

In [2]:
if not DATA_YAML.is_file():
    raise FileNotFoundError(DATA_YAML)
if not SAMPLES_JSONL.is_file():
    raise FileNotFoundError(SAMPLES_JSONL)

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8")) or {}
raw_names = cfg.get("names", {})
if isinstance(raw_names, list):
    CLASS_NAMES = {index: str(name) for index, name in enumerate(raw_names)}
else:
    CLASS_NAMES = {int(index): str(name) for index, name in raw_names.items()}

KPT_COUNT, KPT_DIMS = map(int, cfg["kpt_shape"])
if (KPT_COUNT, KPT_DIMS) != (4, 3):
    raise ValueError(f"expected kpt_shape [4, 3], got {cfg['kpt_shape']}")
if not CLASS_NAMES:
    raise ValueError("yolo_pose.yaml#names is empty")

annotations_dir = DATASET_DIR / "annotations"
if not annotations_dir.is_dir():
    raise FileNotFoundError(annotations_dir)


def prepare_training_view() -> Path:
    view_dir = MODEL_ROOT / "dataset"
    view_dir.mkdir(parents=True, exist_ok=True)
    for name, target in {"images": DATASET_DIR / "images", "labels": annotations_dir}.items():
        link = view_dir / name
        relative_target = Path(os.path.relpath(target, link.parent))
        if link.is_symlink():
            if link.readlink() != relative_target:
                raise ValueError(f"unexpected symlink target: {link}")
        elif link.exists():
            raise FileExistsError(link)
        else:
            link.symlink_to(relative_target, target_is_directory=True)
    train_yaml = view_dir / "yolo_pose.yaml"
    train_cfg = dict(cfg)
    train_cfg.pop("path", None)
    train_yaml.write_text(
        yaml.safe_dump(train_cfg, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    return train_yaml

print(f"classes: {CLASS_NAMES}")
print(f"kpt_shape: {[KPT_COUNT, KPT_DIMS]}")

classes: {0: 'SFP_00', 1: 'SFP_01', 2: 'SFP_10', 3: 'SFP_11', 4: 'SFP_20', 5: 'SFP_21', 6: 'SFP_30', 7: 'SFP_31', 8: 'SFP_40', 9: 'SFP_41', 10: 'sc_port'}
kpt_shape: [4, 3]


## Dataset validation

학습 전에 split 경로, image/annotation 1:1 대응, 17-field YOLO pose row, class ID, 정규화 좌표, visibility, `samples.jsonl` 참조를 검사합니다. 빈 annotation은 정상 negative sample입니다.

In [3]:
def split_dir(split: str) -> Path:
    value = Path(str(cfg[split]))
    if value.is_absolute():
        raise ValueError(f"{split} path must be relative: {value}")
    return DATASET_DIR / value


def iter_images(path: Path) -> list[Path]:
    return sorted(
        candidate
        for candidate in path.rglob("*")
        if candidate.is_file() and candidate.suffix.lower() in IMAGE_EXTS
    )


def annotation_for(image_path: Path) -> Path:
    relative = image_path.relative_to(DATASET_DIR / "images")
    return (annotations_dir / relative).with_suffix(".txt")


def validate_annotation(path: Path) -> list[str]:
    errors = []
    expected_fields = 5 + KPT_COUNT * KPT_DIMS
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) != expected_fields:
            errors.append(f"{path}:{line_number}: expected {expected_fields} fields, got {len(fields)}")
            continue
        try:
            class_id = int(fields[0])
            values = [float(value) for value in fields[1:]]
        except ValueError:
            errors.append(f"{path}:{line_number}: invalid number")
            continue
        if class_id not in CLASS_NAMES:
            errors.append(f"{path}:{line_number}: unknown class_id {class_id}")
        bbox = values[:4]
        if any(value < 0.0 or value > 1.0 for value in bbox):
            errors.append(f"{path}:{line_number}: bbox outside [0, 1]")
        if bbox[2] <= 0.0 or bbox[3] <= 0.0:
            errors.append(f"{path}:{line_number}: bbox width/height must be positive")
        for index in range(4, len(values), KPT_DIMS):
            x, y, visibility = values[index:index + 3]
            if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
                errors.append(f"{path}:{line_number}: keypoint outside [0, 1]")
            if visibility not in (0.0, 1.0, 2.0):
                errors.append(f"{path}:{line_number}: visibility must be 0, 1, or 2")
    return errors


errors = []
dataset_images = set()
split_counts = {}
for split in ("train", "val", "test"):
    if split not in cfg:
        continue
    image_root = split_dir(split)
    images = iter_images(image_root)
    split_counts[split] = len(images)
    dataset_images.update(path.relative_to(DATASET_DIR).as_posix() for path in images)
    if split in ("train", "val") and not images:
        errors.append(f"{split}: no images under {image_root}")
    expected_annotations = {annotation_for(path) for path in images}
    missing = [path for path in expected_annotations if not path.is_file()]
    annotation_root = annotations_dir / split
    actual_annotations = set(annotation_root.rglob("*.txt")) if annotation_root.exists() else set()
    orphan = actual_annotations - expected_annotations
    errors.extend(f"missing annotation: {path}" for path in sorted(missing))
    errors.extend(f"orphan annotation: {path}" for path in sorted(orphan))
    for annotation_path in sorted(actual_annotations):
        errors.extend(validate_annotation(annotation_path))

sample_images = set()
connector_counts = Counter()
camera_counts = Counter()
for line_number, line in enumerate(SAMPLES_JSONL.read_text(encoding="utf-8").splitlines(), 1):
    try:
        row = json.loads(line)
    except json.JSONDecodeError as exc:
        errors.append(f"samples.jsonl:{line_number}: {exc}")
        continue
    connector_counts[str(row.get("connector", "unknown"))] += 1
    for camera, relative_image in row.get("images", {}).items():
        sample_images.add(relative_image)
        camera_counts[camera] += 1
        if not (DATASET_DIR / relative_image).is_file():
            errors.append(f"samples.jsonl:{line_number}: missing image {relative_image}")
        relative_annotation = row.get("annotations", {}).get(camera)
        if relative_annotation is None or not (DATASET_DIR / relative_annotation).is_file():
            errors.append(f"samples.jsonl:{line_number}: missing annotation for {camera}")

errors.extend(f"image missing from samples.jsonl: {path}" for path in sorted(dataset_images - sample_images))
errors.extend(f"samples.jsonl references unknown image: {path}" for path in sorted(sample_images - dataset_images))

print(f"split images: {split_counts}")
print(f"connectors: {dict(connector_counts)}")
print(f"cameras: {dict(camera_counts)}")
print(f"classes: {CLASS_NAMES}")
if errors:
    preview = "\n".join(errors[:100])
    raise ValueError(f"Dataset validation failed with {len(errors)} errors.\n{preview}")
print(f"OK: images={len(dataset_images)}, expected label fields={5 + KPT_COUNT * KPT_DIMS}")

split images: {'train': 6219, 'val': 1362, 'test': 1337}
connectors: {'SFP': 2973}
cameras: {'left': 2973, 'center': 2972, 'right': 2973}
classes: {0: 'SFP_00', 1: 'SFP_01', 2: 'SFP_10', 3: 'SFP_11', 4: 'SFP_20', 5: 'SFP_21', 6: 'SFP_30', 7: 'SFP_31', 8: 'SFP_40', 9: 'SFP_41', 10: 'sc_port'}
OK: images=8918, expected label fields=17


## Train

`classes` 인자를 넘기지 않으므로 YAML에 선언된 SFP와 SC class를 모두 학습합니다. Corner 의미 보존을 위해 horizontal/vertical flip은 비활성화합니다. `TRAIN_DEVICE=[0, 1]`이면 Ultralytics DDP로 Kaggle T4 두 개를 사용합니다.

In [ ]:
from ultralytics import YOLO

TRAIN_YAML = prepare_training_view()
model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(MODEL_ROOT),
    name=RUN_NAME,
    patience=PATIENCE,
    save=True,
    save_period=10,
    plots=True,
    device=TRAIN_DEVICE,
    workers=WORKERS,
    fliplr=0.0,
    flipud=0.0,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
print(f"run: {RUN_DIR}")
print(f"best: {BEST_PT}")

## Resume or validate

필요한 셀만 주석을 해제해 실행합니다.

In [ ]:
# 완전 재개
# model = YOLO(str(LAST_PT))
# train_results = model.train(resume=True)

# 학습 없이 validation
# model = YOLO(str(BEST_PT))
# metrics = model.val(
#     data=str(TRAIN_YAML), imgsz=IMGSZ, batch=BATCH, device=INFERENCE_DEVICE
# )
# print(metrics)

## Prediction preview

In [ ]:
import matplotlib.pyplot as plt

preview_images = iter_images(split_dir("val")) or iter_images(split_dir("train"))
sample = random.choice(preview_images)
model = YOLO(str(BEST_PT))
prediction = model.predict(
    source=str(sample),
    imgsz=IMGSZ,
    device=INFERENCE_DEVICE,
    save=True,
    project=str(MODEL_ROOT),
    name=f"{RUN_NAME}_predict",
)
annotated_bgr = prediction[0].plot()
plt.figure(figsize=(10, 8))
plt.imshow(annotated_bgr[..., ::-1])
plt.axis("off")
plt.title(sample.name)
print(sample)